# einops-reduce-min — worked example 3: 3x3 min-pool a feature map using patch decomposition

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-reduce-min`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

By decomposing a spatial axis into a block coordinate and a patch coordinate — `(h p1)` — and then reducing over the patch axes with `'min'`, you get non-overlapping min-pooling. The pattern `'b c (h p1) (w p2) -> b c h w'` with `p1=3, p2=3` divides each `3x3` patch and keeps the minimum, downsampling from `H x W` to `H/3 x W/3`.

## Worked solution

Input: `(B=1, C=2, H=6, W=9)`. We want 3x3 min pooling: output `(1, 2, 2, 3)`.

**Pattern:** `'b c (h p1) (w p2) -> b c h w'` with `p1=3, p2=3, reduction='min'`.

**Step 1.** `H=6` splits into `h=2` groups, each of `p1=3` rows. `W=9` splits into `w=3` groups, each of `p2=3` columns.
**Step 2.** For each `(b, c, h, w)` output cell, einops takes the min over the `3x3` patch at `(3h:3h+3, 3w:3w+3)` in the input.
**Step 3.** Output shape: `(1, 2, 2, 3)`.

In [ ]:
import torch as t
from einops import reduce

t.manual_seed(60)
B, C, H, W = 2, 3, 6, 9
x = t.randn(B, C, H, W)

def min_pool_3x3(x):
    return reduce(x, 'b c (h p1) (w p2) -> b c h w', 'min', p1=3, p2=3)

out = min_pool_3x3(x)
print('Input shape:', x.shape)
print('Output shape:', out.shape)  # (2, 3, 2, 3)
assert out.shape == (B, C, H // 3, W // 3)

# Verify by manual patch iteration
for b in range(B):
    for c in range(C):
        for h in range(H // 3):
            for w in range(W // 3):
                patch = x[b, c, 3*h:3*h+3, 3*w:3*w+3]
                assert abs(out[b, c, h, w].item() - patch.min().item()) < 1e-6
print('All patch minimums correct:', True)